In [1]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path

datasetPath = Path("../dataset/data_wound_seg_256")

IMG_SIZE = 256

test_images_path = sorted(
    (datasetPath / "test_images").glob("*.png")
)

test_masks_path = sorted(
    (datasetPath / "test_masks").glob("*.png")
)

print("Images:", len(test_images_path))
print("Masks :", len(test_masks_path))

2026-08-23 13:22:48.261037: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-23 13:22:48.286659: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-23 13:22:48.286687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-23 13:22:48.287307: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-23 13:22:48.292045: I tensorflow/core/platform/cpu_feature_guar

Images: 552
Masks : 552


In [2]:
X_test_ds1 = []
Y_test_ds1 = []

for image_path, mask_path in zip(
    test_images_path,
    test_masks_path
):

    image = cv2.imread(str(image_path))

    image = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    image = cv2.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    mask = cv2.imread(
        str(mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    mask = cv2.resize(
        mask,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_NEAREST
    )

    image = image.astype(
        np.float32
    ) / 255.0

    mask = (
        mask > 127
    ).astype(np.float32)

    mask = np.expand_dims(
        mask,
        axis=-1
    )

    X_test_ds1.append(image)
    Y_test_ds1.append(mask)

X_test_ds1 = np.array(X_test_ds1)
Y_test_ds1 = np.array(Y_test_ds1)

print("X:", X_test_ds1.shape)
print("Y:", Y_test_ds1.shape)

X: (552, 256, 256, 3)
Y: (552, 256, 256, 1)


In [3]:
model = tf.keras.models.load_model(
    "../models/unet_dataset2_refined.keras",
    compile=False
)

print("Dataset-2 refined model loaded!")

2026-08-23 13:22:59.493844: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 13:22:59.519160: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-08-23 13:22:59.525282: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Dataset-2 refined model loaded!


In [4]:
predictions = model.predict(
    X_test_ds1,
    batch_size=1,
    verbose=1
)

pred_masks = (
    predictions >= 0.5
).astype(np.float32)

print(
    "Predictions:",
    predictions.shape
)

2026-08-23 13:23:12.987654: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


552/552 [==============================] - 14s 22ms/step
Predictions: (552, 256, 256, 1)


In [5]:
def dice_score(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(
        y_true * y_pred
    )

    return (
        2 * intersection + smooth
    ) / (
        np.sum(y_true)
        + np.sum(y_pred)
        + smooth
    )


def iou_score(
    y_true,
    y_pred,
    smooth=1e-6
):

    y_true = y_true.astype(np.float32)
    y_pred = y_pred.astype(np.float32)

    intersection = np.sum(
        y_true * y_pred
    )

    union = (
        np.sum(y_true)
        + np.sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )

In [6]:
dice_scores = []
iou_scores = []

for i in range(len(Y_test_ds1)):

    true_mask = Y_test_ds1[i].squeeze()
    pred_mask = pred_masks[i].squeeze()

    dice_scores.append(
        dice_score(
            true_mask,
            pred_mask
        )
    )

    iou_scores.append(
        iou_score(
            true_mask,
            pred_mask
        )
    )

dice_scores = np.array(dice_scores)
iou_scores = np.array(iou_scores)

print("Dataset-2 Refined Model → Dataset-1 Test")
print("-----------------------------------------")

print(
    "Test images:",
    len(Y_test_ds1)
)

print(
    "Mean Dice:",
    dice_scores.mean()
)

print(
    "Mean IoU:",
    iou_scores.mean()
)

print(
    "Median Dice:",
    np.median(dice_scores)
)

print(
    "Median IoU:",
    np.median(iou_scores)
)

Dataset-2 Refined Model → Dataset-1 Test
-----------------------------------------
Test images: 552
Mean Dice: 0.7897153503107109
Mean IoU: 0.6982333165099409
Median Dice: 0.8793103450876932
Median IoU: 0.784615385443787
